In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/archive.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/data")

print("Extracted")

Extracted


In [ ]:
import os
print(os.listdir("/content/data/data"))

['train', 'val']


In [ ]:
!pip install -q tensorflow matplotlib seaborn scikit-learn

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
train_dir = "/content/data/data/train"
val_dir   = "/content/data/data/val"

img_size = 224
batch_size = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical'
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

num_classes = train_gen.num_classes

Found 33984 images belonging to 4 classes.
Found 6400 images belonging to 4 classes.


In [ ]:
efficient_base = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

efficient_base.trainable = False

x = GlobalAveragePooling2D()(efficient_base.output)
x = BatchNormalization()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(num_classes, activation='softmax')(x)

efficientnet_model = Model(efficient_base.input, output)

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

efficientnet_model.summary()

In [ ]:
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3)
]

history1 = efficientnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=4,
    callbacks=callbacks
)

Epoch 1/4
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 443s 389ms/step - accuracy: 0.6156 - loss: 0.9235 - val_accuracy: 0.6383 - val_loss: 0.7863 - learning_rate: 0.0010
Epoch 2/4
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 395s 372ms/step - accuracy: 0.6989 - loss: 0.6896 - val_accuracy: 0.6692 - val_loss: 0.7430 - learning_rate: 0.0010
Epoch 3/4
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 402s 378ms/step - accuracy: 0.7303 - loss: 0.6186 - val_accuracy: 0.6791 - val_loss: 0.7730 - learning_rate: 0.0010
Epoch 4/4
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 409s 385ms/step - accuracy: 0.7528 - loss: 0.5722 - val_accuracy: 0.6870 - val_loss: 0.7622 - learning_rate: 0.0010


In [ ]:
# Unfreeze last 120 layers (stronger adaptation)
for layer in efficient_base.layers[-120:]:
    layer.trainable = True

efficientnet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=1, factor=0.3)
]

history2 = efficientnet_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=callbacks
)

Epoch 1/5
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 489s 415ms/step - accuracy: 0.6241 - loss: 0.8529 - val_accuracy: 0.6194 - val_loss: 0.8285 - learning_rate: 2.0000e-05
Epoch 2/5
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 437s 412ms/step - accuracy: 0.7325 - loss: 0.6007 - val_accuracy: 0.6520 - val_loss: 0.7902 - learning_rate: 2.0000e-05
Epoch 3/5
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 432s 406ms/step - accuracy: 0.8001 - loss: 0.4701 - val_accuracy: 0.7270 - val_loss: 0.6229 - learning_rate: 2.0000e-05
Epoch 4/5
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 437s 411ms/step - accuracy: 0.8448 - loss: 0.3714 - val_accuracy: 0.7895 - val_loss: 0.4783 - learning_rate: 2.0000e-05
Epoch 5/5
1062/1062 ━━━━━━━━━━━━━━━━━━━━ 430s 405ms/step - accuracy: 0.8847 - loss: 0.2867 - val_accuracy: 0.7845 - val_loss: 0.5136 - learning_rate: 2.0000e-05


In [ ]:
loss, acc = efficientnet_model.evaluate(val_gen)
print("EfficientNetB0 Validation Accuracy:", acc * 100)

200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - accuracy: 0.7895 - loss: 0.4783
EfficientNetB0 Validation Accuracy: 78.95312309265137
